# Example 4: Creating a Coloured Globe (Complete Pipeline)

This notebook demonstrates the complete `globe3d` model generation pipeline:
1. Generate high-resolution outer sphere and inner sphere.
2. Apply ETOPO topography displacement to the outer shell.
3. Apply sharp coastline step boundary from a shapefile.
4. Split and hollow both hemispheres while adding magnet joint features.
5. Assign vertex colors to the outer shell using a seismic tomography dataset.
6. Assign a solid, neutral gray color to the inner hollow shell vertices using a color modifier.
7. Export both hemispheres to OBJ files with vertex colors.

## Step 1: Import libraries

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import trimesh
from globe3d import (
    generate_sphere_points_fibonacci,
    load_netcdf_grid,
    calculate_displacement_scale,
    displace_vertices,
    displace_near_lines,
    create_hollow_hemispheres,
    assign_vertex_colors,
    modify_vertex_colors,
    write_obj_with_vertex_colors
)

## Step 2: Generate outer and inner spheres

In [ ]:
model_radius_mm = 40.0
outer_v, outer_f = generate_sphere_points_fibonacci(n_points=8000, radius=model_radius_mm)
inner_v, inner_f = generate_sphere_points_fibonacci(n_points=2000, radius=model_radius_mm * 0.75)

## Step 3: Apply topography and coastline step

In [ ]:
# Load topo
topo_lats, topo_lons, topo_grid = load_netcdf_grid("../inputs/ETOPO_2022_v1_60s_N90W180_surface.nc", 'lat', 'lon', 'z')
topo_lats_ds, topo_lons_ds, topo_grid_ds = topo_lats[::10], topo_lons[::10], topo_grid[::10, ::10]

# Displace topography
scale = calculate_displacement_scale(model_radius_mm, earth_radius_km=6371.0, vertical_exagg=40.0)
outer_v = displace_vertices(outer_v, topo_lats_ds, topo_lons_ds, topo_grid_ds, scale)

# Coastline step
outer_v = displace_near_lines(
    vertices=outer_v,
    shapefile_path="../inputs/coastlines/ne_110m_coastline.shp",
    displacement=0.8,
    width_degrees=0.5
)

## Step 4: Split, hollow, and insert magnets

In [ ]:
magnet_params = {
    'magnet_diameter': 5.0,
    'magnet_height': 2.0,
    'h_tol': 0.15,
    'v_tol': 0.10,
    'v_offset': 0.20,
    'min_thick': 1.5,
    'n_magnets': 3,
    'add_bosses': True
}

top_half, bottom_half = create_hollow_hemispheres(
    outer_vertices=outer_v,
    outer_faces=outer_f,
    inner_vertices=inner_v,
    inner_faces=inner_f,
    engine='manifold',
    magnet_params=magnet_params
)

## Step 5: Apply independent surface coloring

We color the outer shell from a seismic tomography grid, and then we use the `modify_vertex_colors` modifier function with the `'inward_facing'` selection option to paint the inner cavity vertices a solid grey.

In [ ]:
# Load tomography grid
tomo_lats, tomo_lons, tomo_grid = load_netcdf_grid("../inputs/s40_depth_slice_2850.grd", 'y', 'x', 'z')

# 1. Color top hemisphere using tomography grid
top_colors = assign_vertex_colors(
    vertices=top_half.vertices,
    lats=tomo_lats,
    lons=tomo_lons,
    grid=tomo_grid,
    colormap='RdBu_r',
    vmin=-2.0,
    vmax=2.0
)

# 2. Recolor top hemisphere inner cavity to gray
top_colors = modify_vertex_colors(
    vertices=top_half.vertices,
    colors=top_colors,
    selection_function='inward_facing',
    faces=top_half.faces,
    constant_color=[0.6, 0.6, 0.6]
)

# 3. Color bottom hemisphere using tomography grid
bottom_colors = assign_vertex_colors(
    vertices=bottom_half.vertices,
    lats=tomo_lats,
    lons=tomo_lons,
    grid=tomo_grid,
    colormap='RdBu_r',
    vmin=-2.0,
    vmax=2.0
)

# 4. Recolor bottom hemisphere inner cavity to gray
bottom_colors = modify_vertex_colors(
    vertices=bottom_half.vertices,
    colors=bottom_colors,
    selection_function='inward_facing',
    faces=bottom_half.faces,
    constant_color=[0.6, 0.6, 0.6]
)

## Step 6: Preview colors in 3D

In [ ]:
fig = plt.figure(figsize=(12, 6))

# Top half colored
ax1 = fig.add_subplot(121, projection='3d')
pts_top = top_half.vertices
sc1 = ax1.scatter(pts_top[:, 0], pts_top[:, 1], pts_top[:, 2], c=top_colors, s=2)
ax1.set_title("Colored Top Hemisphere")

# Bottom half colored
ax2 = fig.add_subplot(122, projection='3d')
pts_bot = bottom_half.vertices
sc2 = ax2.scatter(pts_bot[:, 0], pts_bot[:, 1], pts_bot[:, 2], c=bottom_colors, s=2)
ax2.set_title("Colored Bottom Hemisphere")

plt.show()

## Step 7: Export to OBJ with vertex colors

In [ ]:
output_dir = "../outputs"
os.makedirs(output_dir, exist_ok=True)

write_obj_with_vertex_colors(
    filename=os.path.join(output_dir, "example_4_top.obj"),
    vertices=top_half.vertices,
    faces=top_half.faces,
    colors=top_colors
)

write_obj_with_vertex_colors(
    filename=os.path.join(output_dir, "example_4_bottom.obj"),
    vertices=bottom_half.vertices,
    faces=bottom_half.faces,
    colors=bottom_colors
)
print("Colored OBJ hemispheres exported successfully!")